In [1]:
#  Se instalan librerias (cada vez en sesión de Colab)
!pip install supabase python-dotenv pandas numpy -q
# Supabase es un aplicativo web para alojar BD que usa postgresql y tiene un almacenamiento gratuito que para el ejercicio es suficiente
# python-dotenv es una libreria para extraer credenciales de manera segura desde un archivo .env

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.1 MB/s eta 0:00:00


In [1]:
# Se cargan credenciales de forma segura desde .env
# Se sube un archivo .env a Colab con:
#   SUPABASE_URL= "https://xyzxyzxyzyxz.supabase.co"
#   SUPABASE_KEY= key_de_supabase

import os
from dotenv import load_dotenv
from supabase import create_client

# load_dotenv(dotenv_path='/content/drive/MyDrive/.env')
load_dotenv()  # Si tienes el .env directamente en la carpeta de trabajo

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError(
        "Faltan SUPABASE_URL / SUPABASE_KEY. Verifica que subiste el .env "
        "a la carpeta de trabajo de Colab (/content/.env)."
    )

# Crea la conexion entre la sesion de Colab y Supabase
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [12]:
%pip install -e ..

Obtaining file:///G:/Mi%20unidad/Elecciones2_2026_PDFs/e14
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for e14 (pyproject.toml): started
  Building editable for e14 (pyproject.toml): finished with status 'done'
  Created wheel for e14: filename=e14-0.1.0-0.editable-py3-none-any.whl size=1150 sha256=a21f1f298e533274fe7048b1e747283fa7aa0a7d5afdb3b04b1d1dc8cec1b5eb
  Stored in directory: C:\Users\miggi\AppData\Local\Temp\pip-ephem-wheel-cache-f5zcjxp_\wheels\1f\24\ad\72eb9aa62a3f5dfd5a4f29c15efd201b15807cce4e158ce12c
Successf


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Se extrae la tabla DIVIPOLE desde Supabase (solo columnas necesarias)
import pandas as pd
from ingestion.extract_e14_sample import fetch_all_rows, get_stratum, get_sample_size, sample_voting_tables

TABLE_NAME = "divipole_regis"  # <-- Este es el nombre que se le dio a la tabla en Supabase
COLUMNS = "dd,mm,zz,pp,departamento,municipio,puesto,comuna,mesas_domingo"



rows = fetch_all_rows(supabase, TABLE_NAME, COLUMNS)
df = pd.DataFrame(rows)
print(f"Puestos extraídos de Supabase: {len(df)}")
df.head()


Puestos extraídos de Supabase: 13742


,dd,mm,zz,pp,departamento,municipio,puesto,comuna,mesas_domingo
0,01,001,01,01,ANTIOQUIA,MEDELLIN,SEC. ESC. LA ESPERANZA No 2,01COMUNA 1\nPOPULAR,36
1,01,001,01,02,ANTIOQUIA,MEDELLIN,INST.EDUC. LA CANDELARIA,01COMUNA 1\nPOPULAR,44
2,01,001,01,03,ANTIOQUIA,MEDELLIN,IE MARIA DE LOS ANGELES CANO MARQUEZ,01COMUNA 1\nPOPULAR,25
3,01,001,01,04,ANTIOQUIA,MEDELLIN,SEC. ESC. MEDELLIN,01COMUNA 1\nPOPULAR,26
4,01,001,01,05,ANTIOQUIA,MEDELLIN,I.E.FE Y ALEGRIA GRANIZAL,01COMUNA 1\nPOPULAR,15


In [3]:
# Se excluyen consulados (dd = 88)
# Puesto que no hay imágenes de formularios del exterior en la pagina
# https://escrutinios2vueltapresidente2026.registraduria.gov.co/actas-e14,
# por lo cual no aportan al universo de entrenamiento/validación del OCR.
df = df[df["dd"] != "88"].copy()
print(f"Puestos tras excluir consulados (dd=88): {len(df)}")
print(f"Total mesas en universo doméstico: {df['mesas_domingo'].sum()}")

Puestos tras excluir consulados (dd=88): 13489
Total mesas en universo doméstico: 118346


In [4]:

# Asignamos zonas y se crea nueva columna "estratos" con base en la definicion de zona
# Dado que se quiere tomar muestra de todas las posibles condiciones de digitalizacion
# Las diferentes zonas tendrian posibles diferencias de equipos de digitalizacion,
# problemas de transporte y acceso, etc.
# Se tuvo en cuenta la definicion que tiene la registraduria por la codificacion de las zonas
#   zz = 99             -> RURAL (veredas / corregimientos), en TODOS los municipios,
#                          incluidos los zonificados (ej. corregimientos de Medellín).
#   zz = 98             -> CARCEL (entorno de captura distinto, se trata aparte).
#   zz = 00 o 01..97    -> URBANO. 00 es la zona por defecto de los ~1.100 municipios
#                          NO zonificados (su única cabecera). 01-97 son números de
#                          comuna reales, pero solo existen en los municipios
#                          "zonificados" (Bogotá, Medellín, Cali, Barranquilla, etc.).
df["estrato"] = df["zz"].apply(get_stratum)


In [5]:
# Se calcula el Universo total de mesas N
N = df["mesas_domingo"].sum()

print(f"Universo total de mesas (segunda vuelta, sin consulados): {N}")

# Se calcula el tamaño de muestra (n) con la fórmula de Cochran con corrección por población finita
# Se asume Probabilidad p y q de 50% por que no se conoce su valor

CONFIANZA = 0.95
MARGEN_ERROR = 0.05

n = get_sample_size(N, confidence=CONFIANZA, margin=MARGEN_ERROR)
print(f"Tamaño de muestra (n) requerido ({int(CONFIANZA*100)}% / ±{int(MARGEN_ERROR*100)}%): {n}")


Universo total de mesas (segunda vuelta, sin consulados): 118346
Tamaño de muestra (n) requerido (95% / ±5%): 383


In [6]:
# Se calcula la proporcion de mesas por estrato en total

print("# --------------- # Proporcion de mesas por estrato en porcentaje: ")
print((df.groupby("estrato")["mesas_domingo"].sum() / df["mesas_domingo"].sum()).round(4))

print("\n# --------------- # Proporcion de mesas por estrato: ")
stratum_table_counts = df.groupby("estrato")["mesas_domingo"].sum()
print("Mesas por estrato:")
print(stratum_table_counts)

# --------------- # Proporcion de mesas por estrato en porcentaje: 
estrato
carcel    0.0014
rural     0.1636
urbano    0.8350
Name: mesas_domingo, dtype: float64

# --------------- # Proporcion de mesas por estrato: 
Mesas por estrato:
estrato
carcel      162
rural     19365
urbano    98819
Name: mesas_domingo, dtype: int64


In [7]:
# Se genera la asignación proporcional por estrato de acuento a la muestra calculada
# (con corrección de redondeo)
stratum_counts = df.groupby("estrato")["mesas_domingo"].sum()
stratum_allocation = (stratum_counts / N * n).round().astype(int)

# Corrige el residuo de redondeo para que la suma cuadre exactamente con n
rounding_diff = n - stratum_allocation.sum()
if rounding_diff != 0:
    higher_stratum = stratum_counts.idxmax()  # Devuelve el indice con valor mas alto
    stratum_allocation[higher_stratum] += rounding_diff   # Se suma o resta la diferencia

print("Mesas a muestrear por estrato:")
print(stratum_allocation)


Mesas a muestrear por estrato:
estrato
carcel      1
rural      63
urbano    319
Name: mesas_domingo, dtype: int64


In [8]:
sample_df = sample_voting_tables(df, stratum_allocation)
print(f"Total mesas muestreadas: {len(sample_df)}")
print(sample_df["estrato"].value_counts())

Total mesas muestreadas: 383
estrato
urbano    319
rural      63
carcel      1
Name: count, dtype: int64


In [9]:
# Formato tabular, útil para cruzar contra tus archivos de imágenes escaneadas
sample_df.to_csv("../data/muestra_e14_segunda_vuelta.csv", index=False, encoding="utf-8-sig")
print("Guardado: muestra_e14_segunda_vuelta.csv")



Guardado: muestra_e14_segunda_vuelta.csv


In [10]:
from scraper.scraper_e14 import execute_scraper

#Se importa y genera el llamado a la funcion execute_scraper()

In [ ]:
execute_scraper()

[Selección] 367 puesto(s) único(s) | 383 mesa(s) a descargar

[=== FASE 1: DESCARGA DE ESTRUCTURAS MAESTRAS ===]
  [Caché] Leyendo localmente: index.json
  [Caché] Leyendo localmente: divipole.json

[=== FASE 2: RESOLUCIÓN DE RUTAS (SOLO PUESTOS SELECCIONADOS) ===]
[*] Puesto seleccionado: 01/001/02/02 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_02_02_mesas_20260621_210742_815.json
[*] Puesto seleccionado: 01/001/06/02 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_06_02_mesas_20260621_192741_771.json
[*] Puesto seleccionado: 01/001/09/05 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_09_05_mesas_20260621_193752_029.json
[*] Puesto seleccionado: 01/001/10/03 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_10_03_mesas_20260621_185606_874.json
[*] Puesto seleccionado: 01/001/11/04 (1 mesa(s))
  [Caché] Leyendo localmente: actas_documentos_001_01_001_11_04_mesas_20260621_183245_345.json
[*] Puest